# 08. Window Calculations & Resampling (5+ Years Interview Guide)
Exhaustive revision guide to rolling statistics, expanding totals, datetime resampling, shift, diff, and pct_change on raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **Rolling & Expanding Windows**: Dedicated cell for `.rolling().mean()` and `.expanding().sum()`.
- **Time-Series Resampling**: Dedicated cell for `df.resample('ME').sum()`.
- **Shifting & Differencing**: Dedicated cell for `.shift()`, `.diff()`, and `.pct_change()`.

This interactive revision guide uses `data/raw_transactions.csv` for all real-world code examples.

In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns


### Moving Average Windows with `.rolling()`
**Explanation**: Calculates rolling 7-day average transaction volume and spend.

**Syntax**: `daily_ts['transaction_amount'].rolling(window=7, min_periods=1).mean()`

In [2]:
daily_ts = (
    df
    .assign(date=pd.to_datetime(df['transaction_date'], format='mixed', errors='coerce'))
    .dropna(subset=['date', 'transaction_amount'])
    .set_index('date')
    .sort_index()
    .resample('D')['transaction_amount'].sum()
)
rolling_7d = daily_ts.rolling(7, min_periods=1).mean()
print('7-Day Rolling Daily Total Spend Head:\n', rolling_7d.head(10))

7-Day Rolling Daily Total Spend Head:
 date
2025-01-01    26310.740000
2025-01-02    28735.910000
2025-01-03    27273.380000
2025-01-04    27826.787500
2025-01-05    26571.246000
2025-01-06    28435.608333
2025-01-07    27959.904286
2025-01-08    27909.820000
2025-01-09    26117.595714
2025-01-10    27008.602857
Freq: D, Name: transaction_amount, dtype: float64


### Cumulative Running Totals with `.expanding()`
**Explanation**: Calculates lifetime running cumulative transaction volume.

**Syntax**: `daily_ts.expanding().sum()`

In [3]:
cumulative_spend = daily_ts.expanding().sum()
print('Lifetime Expanding Cumulative Spend Head:\n', cumulative_spend.head())

Lifetime Expanding Cumulative Spend Head:
 date
2025-01-01     26310.74
2025-01-02     57471.82
2025-01-03     81820.14
2025-01-04    111307.15
2025-01-05    132856.23
Freq: D, Name: transaction_amount, dtype: float64


### Frequency-Based Resampling with `.resample()`
**Explanation**: Aggregates transaction revenue by month-end (`'ME'`) and week (`'W'`).

**Syntax**: `daily_ts.resample('ME').sum()`

In [4]:
monthly_spend = daily_ts.resample('ME').sum()
print('Monthly Total Transaction Spend (resample ME):\n', monthly_spend)

Monthly Total Transaction Spend (resample ME):
 date
2025-01-31    857671.41
2025-02-28    803674.47
2025-03-31    876888.29
2025-04-30    860398.85
2025-05-31    909074.53
2025-06-30    879312.28
2025-07-31    882477.62
2025-08-31    911868.50
2025-09-30    860245.42
2025-10-31    868116.65
2025-11-30    870693.41
2025-12-31    915582.21
2026-01-31    785004.29
2026-02-28    790693.81
2026-03-31    848786.82
2026-04-30    782445.90
2026-05-31    423426.53
2026-06-30     31240.84
2026-07-31     34100.01
2026-08-31     27457.81
2026-09-30     33883.98
2026-10-31     26674.29
2026-11-30     30438.30
2026-12-31     39381.92
Freq: ME, Name: transaction_amount, dtype: float64


### Lagging Time Series with `.shift()`
**Explanation**: Creates previous day's spend feature for time-series forecasting.

**Syntax**: `daily_ts.shift(1)`

In [5]:
lagged_spend = daily_ts.shift(1)
print('Lagged Daily Spend (t-1) Head:\n', lagged_spend.head())

Lagged Daily Spend (t-1) Head:
 date
2025-01-01         NaN
2025-01-02    26310.74
2025-01-03    31161.08
2025-01-04    24348.32
2025-01-05    29487.01
Freq: D, Name: transaction_amount, dtype: float64


### First-Order Differencing with `.diff()`
**Explanation**: Calculates day-over-day spend acceleration.

**Syntax**: `daily_ts.diff(1)`

In [6]:
spend_diff = daily_ts.diff(1)
print('Day-over-Day Spend Delta Head:\n', spend_diff.head())

Day-over-Day Spend Delta Head:
 date
2025-01-01        NaN
2025-01-02    4850.34
2025-01-03   -6812.76
2025-01-04    5138.69
2025-01-05   -7937.93
Freq: D, Name: transaction_amount, dtype: float64


### Percentage Growth with `.pct_change()`
**Explanation**: Calculates daily growth rate in transaction volume.

**Syntax**: `daily_ts.pct_change(1) * 100`

In [7]:
spend_growth = daily_ts.pct_change(1) * 100
print('Daily Transaction Spend Growth (%):\n', spend_growth.dropna().head())

Daily Transaction Spend Growth (%):
 date
2025-01-02    18.434829
2025-01-03   -21.863042
2025-01-04    21.104906
2025-01-05   -26.920091
2025-01-06    75.215926
Name: transaction_amount, dtype: float64


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Rolling Volatility in Daily Revenue
**Explanation**: Calculate the 30-day rolling standard deviation of daily transaction revenue to monitor revenue stability.

**Syntax**: `daily_ts.rolling(30).std()`

In [8]:
rolling_vol_30d = daily_ts.rolling(30).std()
print('30-Day Rolling Revenue Volatility (Std Dev):\n', rolling_vol_30d.dropna().head())

30-Day Rolling Revenue Volatility (Std Dev):
 date
2025-01-30    4723.596457
2025-01-31    4995.937467
2025-02-01    4972.723387
2025-02-02    5034.378463
2025-02-03    5092.628682
Freq: D, Name: transaction_amount, dtype: float64
